In [7]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.document_loaders import TextLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

In [2]:
loader = DirectoryLoader(r'C:\Users\mendh\Langchain-Langgraph\0-Dataingestion-parsing\data\text_files', glob='**/*.txt', show_progress=True,loader_cls=TextLoader)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)
print(f"Number of documents: {len(docs)}")

100%|██████████| 4/4 [00:00<00:00, 2196.83it/s]

Number of documents: 14


In [3]:
retriver = FAISS.from_documents(docs, HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")).as_retriever(search_type="mmr",search_kwargs={"k": 8})


C:\Users\mendh\AppData\Local\Temp\ipykernel_5400\2726565219.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  retriver = FAISS.from_documents(docs, HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")).as_retriever(search_type="mmr",search_kwargs={"k": 8})


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
llm = init_chat_model(model_provider="groq", model="llama-3.1-8b-instant")
llm.invoke("Hello, how are you?")

AIMessage(content="I'm functioning properly. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 41, 'total_tokens': 54, 'completion_time': 0.021636487, 'completion_tokens_details': None, 'prompt_time': 0.003288585, 'prompt_tokens_details': None, 'queue_time': 0.055316944, 'total_time': 0.024925072}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d435f-7c55-7fe2-95dc-31b6a40e5b07-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 13, 'total_tokens': 54})

In [11]:
prompt = PromptTemplate.from_template("""Answer the question based on the following context: {context}
                                      Question: {input}""")

chain = LLMChain(llm=llm, prompt=prompt)
chain.invoke({"input": "What is Machine Learning?", "context": retriver.invoke("What is Machine Learning?")})

{'input': 'What is Machine Learning?',
 'context': [Document(id='0ad621f0-209e-42e9-98e9-661b7054caa6', metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt'}, page_content='From a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be described as empirical risk minimisation under this framework.\n\nHistory\nSee also: Timeline of machine learning\nThe term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]'),
  Document(id='b9991c29-1cf0-4053-abc4-787ef2c1a06f', metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt